# Mental Health Text Classification using NLP

**Module:** IBM3201 — Data Mining & Predictive Analytics  
**Student:** Suruthi Kattampalayam Sivasankar (I26039351)  
**Session:** January 2026  

---

## Project Summary

This notebook implements a 7-class mental health text classification system trained on Reddit posts. The goal is to automatically detect mental health conditions from informal social media text.

**Classes:** Normal · Depression · Suicidal · Anxiety · Bipolar · Stress · Personality Disorder

**Pipeline overview:**
1. Data Loading & Exploration
2. Exploratory Data Analysis (EDA)
3. NLP Preprocessing (6-step pipeline)
4. Feature Extraction (TF-IDF)
5. Model Training (Naive Bayes · Logistic Regression · SVM)
6. Evaluation (Accuracy · F1-Score · Confusion Matrices)

**Dataset:** [Sentiment Analysis for Mental Health — Kaggle](https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health) (Sarkar, 2023)  
**Methodology:** CRISP-DM

---
## Section 1 — Environment Setup

Install any packages not available by default in Colab, then import all libraries used throughout the notebook.

In [ ]:
# Install wordcloud (not pre-installed in Colab)
!pip install wordcloud -q

In [ ]:
# --- Standard library ---
import os
import re
import time
import zipfile

# --- Data handling ---
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# --- NLP ---
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# --- Machine learning ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully.")

---
## Section 2 — Data Loading

Upload and extract the dataset from Kaggle. The dataset is a ZIP file containing `Combined Data.csv` — 53,043 Reddit posts labelled with mental health categories.

In [ ]:
# Upload the dataset ZIP file from your local machine
from google.colab import files
uploaded = files.upload()

In [ ]:
# Extract the ZIP and confirm the CSV is present
with zipfile.ZipFile('Combined Data.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/mental_health')

print("Extracted files:", os.listdir('/content/mental_health'))

In [ ]:
# Load into a DataFrame and inspect the basic structure
df = pd.read_csv('/content/mental_health/Combined Data.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['status'].value_counts())

In [ ]:
# Preview 5 sample posts to understand the text style
print("=== SAMPLE POSTS ===\n")
for i in range(5):
    print(f"Status    : {df['status'].iloc[i]}")
    print(f"Statement : {df['statement'].iloc[i]}")
    print("-" * 60)

---
## Section 3 — Exploratory Data Analysis (EDA)

Five visualisations to understand the dataset before modelling:

1. Class distribution (bar chart)
2. Percentage breakdown (pie chart)
3. Post text length distribution (histogram)
4. Average post length by category (bar chart)
5. Most frequent words across all posts (word cloud)

> **Key findings:** The dataset is heavily imbalanced — Normal (30.8%) and Depression (29.0%) dominate, while Personality Disorder accounts for just 2.3%. Bipolar and Depression posts are significantly longer than Normal posts, and emotional vocabulary ("feel", "want", "die") dominates the corpus.

In [ ]:
# Consistent colour palette used across all EDA charts
COLORS = ['#2ecc71', '#e74c3c', '#e67e22', '#3498db', '#9b59b6', '#1abc9c', '#f39c12']

In [ ]:
# --- EDA 1: Class Distribution Bar Chart ---
counts = df['status'].value_counts()

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=counts.index, y=counts.values, palette=COLORS)

# Annotate each bar with its count
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height()):,}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.title('Distribution of Mental Health Categories', fontsize=16, fontweight='bold')
plt.xlabel('Mental Health Status', fontsize=12)
plt.ylabel('Number of Posts', fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('v1_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# --- EDA 2: Percentage Breakdown Pie Chart ---
plt.figure(figsize=(9, 9))
plt.pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=COLORS,
    explode=[0.05] * len(counts),
    startangle=140,
    textprops={'fontsize': 11}
)
plt.title('Percentage Distribution of Mental Health Categories', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('v2_pie_chart.png', dpi=150)
plt.show()

In [ ]:
# --- EDA 3: Post Text Length Distribution ---
# Compute character count for each post
df['text_length'] = df['statement'].astype(str).apply(len)

plt.figure(figsize=(10, 5))
plt.hist(df['text_length'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
plt.axvline(
    df['text_length'].mean(), color='red', linestyle='--', linewidth=2,
    label=f"Mean: {df['text_length'].mean():.0f} chars"
)
plt.title('Distribution of Post Text Lengths', fontsize=15, fontweight='bold')
plt.xlabel('Number of Characters', fontsize=12)
plt.ylabel('Number of Posts', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig('v3_text_length.png', dpi=150)
plt.show()

print(f"Mean post length : {df['text_length'].mean():.0f} characters")
print(f"Median           : {df['text_length'].median():.0f} characters")
print(f"Max              : {df['text_length'].max():,} characters")

In [ ]:
# --- EDA 4: Average Post Length by Mental Health Category ---
avg_length = df.groupby('status')['text_length'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=avg_length.index, y=avg_length.values, palette=COLORS)
plt.title('Average Post Length by Mental Health Category', fontsize=15, fontweight='bold')
plt.xlabel('Mental Health Status', fontsize=12)
plt.ylabel('Average Characters', fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('v4_avg_length.png', dpi=150)
plt.show()

print(avg_length.to_string())

In [ ]:
# --- EDA 5: Word Cloud — Most Frequent Terms Across All Posts ---
all_text = ' '.join(df['statement'].astype(str).tolist())

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    colormap='RdYlGn',
    max_words=150,
    collocations=False
).generate(all_text)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Mental Health Posts', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('v5_wordcloud.png', dpi=150)
plt.show()

---
## Section 4 — NLP Preprocessing Pipeline

Raw social media text cannot be fed directly into a machine learning model. This 6-step pipeline transforms noisy, informal text into clean, numerical-ready input.

| Step | Operation | Purpose |
|------|-----------|--------|
| 1 | Drop index column | Remove uninformative metadata |
| 2 | Remove nulls & empty rows | Ensure no blank inputs reach the model |
| 3 | Lowercase conversion | Treat `FEEL`, `Feel`, `feel` as one token |
| 4 | Remove URLs, punctuation, special chars | Strip non-linguistic noise |
| 5 | Tokenization | Split sentences into individual word tokens |
| 6 | Stop word removal + Lemmatization | Remove filler words; reduce words to base forms |

> **Example transformation:**  
> Raw → `"trouble sleeping, confused mind, restless heart. All out of tune"`  
> Clean → `"trouble sleep confused mind restless heart tune"`

In [ ]:
# --- Step 1: Drop the uninformative index column added by Kaggle ---
df = df.drop(columns=['Unnamed: 0'])

print("Remaining columns:", df.columns.tolist())
print("Shape:", df.shape)

In [ ]:
# --- Step 2: Remove null values and empty strings ---
print(f"Before cleaning: {df.shape[0]:,} posts")

df = df.dropna(subset=['statement'])                   # drop NaN rows
df = df[df['statement'].str.strip() != '']             # drop blank strings

print(f"After cleaning : {df.shape[0]:,} posts")
print(f"Removed        : {53043 - df.shape[0]} records")

In [ ]:
# --- Step 3: Convert all text to lowercase ---
df['statement'] = df['statement'].str.lower()

print("Sample after lowercasing:")
print(df['statement'].head(3).to_string())

In [ ]:
# --- Step 4: Remove URLs, punctuation, special characters, and numbers ---
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)          # keep only lowercase letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()      # collapse multiple spaces
    return text

df['statement'] = df['statement'].apply(clean_text)

print("Sample after cleaning:")
print(df['statement'].head(3).to_string())

In [ ]:
# --- Step 5: Tokenization — split each post into a list of word tokens ---
df['tokens'] = df['statement'].apply(word_tokenize)

print("Sample tokens:")
print(df['tokens'].head(3).to_string())

In [ ]:
# --- Step 6: Stop word removal and Lemmatization ---
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_tokens(tokens):
    """Remove stop words and reduce each token to its base (lemma) form."""
    return [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in STOP_WORDS
    ]

df['clean_tokens'] = df['tokens'].apply(preprocess_tokens)
df['clean_text']   = df['clean_tokens'].apply(lambda x: ' '.join(x))

# Show the transformation on a single example
print("BEFORE:", df['tokens'].iloc[1])
print("\nAFTER :", df['clean_tokens'].iloc[1])
print("\nFINAL :", df['clean_text'].iloc[1])

In [ ]:
# --- Pipeline summary ---
print("=" * 60)
print("PREPROCESSING PIPELINE SUMMARY")
print("=" * 60)
print(f"Total posts after cleaning : {len(df):,}")
print(f"\nRAW TEXT EXAMPLE:")
print("  trouble sleeping, confused mind, restless heart. All out of tune")
print(f"\nFINAL CLEAN TEXT:")
print(f"  {df['clean_text'].iloc[1]}")
print("\nDataset ready for feature extraction.")

---
## Section 5 — Feature Extraction (TF-IDF)

Machine learning models cannot read words — they require numerical inputs. **TF-IDF (Term Frequency–Inverse Document Frequency)** converts the cleaned text into a weighted matrix where:

- Common words across all posts (e.g. `feel`) receive **low scores** — they are not discriminative
- Rare but specific words (e.g. `suicidal`) receive **high scores** — they carry predictive signal

**Configuration:**
- `max_features=10,000` — vocabulary capped at the 10,000 most informative terms
- `ngram_range=(1, 2)` — captures both single words and two-word phrases (bigrams), so `not happy` is treated as a single meaningful unit
- **80/20 stratified split** — ensures all 7 classes are proportionally represented in both train and test sets

In [ ]:
# Separate features and labels
X = df['clean_text']
y = df['status']

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves class proportions in both sets
)

# Fit TF-IDF on training data only; transform both sets
tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Training set : {X_train_tfidf.shape[0]:,} posts × {X_train_tfidf.shape[1]:,} features")
print(f"Test set     : {X_test_tfidf.shape[0]:,} posts × {X_test_tfidf.shape[1]:,} features")
print(f"\nSample vocabulary (first 10 terms):")
print(list(tfidf.vocabulary_.keys())[:10])

---
## Section 6 — Model Training

Three classical machine learning classifiers are trained and compared:

| Model | Approach | Strength |
|---|---|---|
| **Multinomial Naive Bayes** | Probabilistic; word–class associations | Fast; good baseline |
| **Logistic Regression** | Calculates probability across all 7 classes | Handles ambiguous posts well |
| **Support Vector Machine** (LinearSVC) | Finds optimal decision boundary in feature space | Fast, competitive accuracy |

All models use `class_weight='balanced'` to compensate for the class imbalance identified during EDA.

In [ ]:
# Define all three models
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),
    'SVM': LinearSVC(
        class_weight='balanced',
        random_state=42,
        max_iter=2000
    )
}

# Train each model, record accuracy, F1-score and training time
results = {}

for name, model in models.items():
    print(f"Training {name}...")

    start  = time.time()
    model.fit(X_train_tfidf, y_train)
    elapsed = time.time() - start

    y_pred = model.predict(X_test_tfidf)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        'model':       model,
        'predictions': y_pred,
        'accuracy':    acc,
        'f1_score':    f1,
        'train_time':  elapsed
    }

    print(f"  Accuracy      : {acc*100:.1f}%")
    print(f"  Weighted F1   : {f1*100:.1f}%")
    print(f"  Training time : {elapsed:.2f}s\n")

---
## Section 7 — Evaluation

Models are evaluated using three complementary metrics:

- **Accuracy** — overall percentage of correct predictions
- **Weighted F1-Score** — balances precision and recall across all 7 classes; preferred for imbalanced datasets
- **Confusion Matrix** — reveals which classes are confused with each other

Three visualisations are produced:
1. Per-class classification reports
2. Side-by-side confusion matrices (row-normalised to %)
3. Bar chart comparison of accuracy, F1-score and training time

In [ ]:
# --- Classification Reports (per-class precision, recall, F1) ---
for name, result in results.items():
    print("=" * 60)
    print(f"{name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_test, result['predictions']))

In [ ]:
# --- Confusion Matrices (row-normalised to percentage) ---
CLASS_LABELS = ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'PD', 'Stress', 'Suicidal']

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['predictions'])
    # Normalise each row to percentages
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    sns.heatmap(
        cm_pct,
        annot=True, fmt='.0f', cmap='Blues',
        xticklabels=CLASS_LABELS,
        yticklabels=CLASS_LABELS,
        ax=axes[idx], cbar=False,
        annot_kws={'size': 11}
    )
    axes[idx].set_title(
        f"{name}\nAccuracy: {result['accuracy']*100:.1f}%",
        fontsize=14, fontweight='bold', pad=15
    )
    axes[idx].set_ylabel('Actual' if idx == 0 else '', fontsize=12)
    axes[idx].set_xlabel('Predicted', fontsize=12)
    axes[idx].tick_params(axis='x', rotation=30, labelsize=11)
    axes[idx].tick_params(axis='y', rotation=0,  labelsize=11)

plt.suptitle('Confusion Matrices — All Models (%)', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout(pad=3.0)
plt.savefig('v6_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Model Performance Comparison Bar Charts ---
model_names = list(results.keys())
acc_scores  = [results[m]['accuracy']   * 100 for m in model_names]
f1_scores   = [results[m]['f1_score']   * 100 for m in model_names]
train_times = [results[m]['train_time']       for m in model_names]

BAR_COLORS = ['#e74c3c', '#3498db', '#2ecc71']
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Accuracy
axes[0].bar(model_names, acc_scores, color=BAR_COLORS, edgecolor='white')
axes[0].set_title('Accuracy (%)', fontsize=13, fontweight='bold')
axes[0].set_ylim(60, 85)
axes[0].set_ylabel('Accuracy (%)')
for i, v in enumerate(acc_scores):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# Weighted F1-Score
axes[1].bar(model_names, f1_scores, color=BAR_COLORS, edgecolor='white')
axes[1].set_title('Weighted F1-Score (%)', fontsize=13, fontweight='bold')
axes[1].set_ylim(60, 85)
axes[1].set_ylabel('F1-Score (%)')
for i, v in enumerate(f1_scores):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# Training Time
axes[2].bar(model_names, train_times, color=BAR_COLORS, edgecolor='white')
axes[2].set_title('Training Time (seconds)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Seconds')
for i, v in enumerate(train_times):
    axes[2].text(i, v + 0.2, f'{v:.2f}s', ha='center', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('v7_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Final results summary table ---
summary = pd.DataFrame([
    {
        'Model':         name,
        'Accuracy (%)':  round(result['accuracy']  * 100, 1),
        'F1-Score (%)':  round(result['f1_score']  * 100, 1),
        'Training Time': f"{result['train_time']:.2f}s"
    }
    for name, result in results.items()
])

print("=" * 55)
print("FINAL MODEL COMPARISON")
print("=" * 55)
print(summary.to_string(index=False))
print("=" * 55)
best = summary.loc[summary['Accuracy (%)'].idxmax(), 'Model']
print(f"\nBest model: {best}")

---
## Section 8 — Conclusions

### Results

| Model | Accuracy | Weighted F1 | Training Time |
|---|---|---|---|
| Naive Bayes | 68.7% | 66.9% | ~0.11s |
| **Logistic Regression** | **76.3%** | **76.3%** | ~19.71s |
| SVM | 75.4% | 75.2% | ~6.06s |

**Logistic Regression** achieved the best overall performance. SVM is a close second and trains ~3× faster — a meaningful trade-off at scale.

### Key Observations

- All three models performed best on the **Normal** class, which dominates the dataset
- **Depression and Suicidal posts were consistently confused** across all models — this reflects a genuine linguistic overlap, not a modelling failure. The language of depression and suicidal ideation is intertwined in ways that are difficult even for humans to separate
- **Stress and Personality Disorder** were the hardest classes, especially for Naive Bayes
- A 76.3% accuracy means roughly 1 in 4 posts is misclassified — not acceptable for clinical deployment, but a meaningful proof of concept

### Limitations

- English-only dataset excludes billions of non-English speakers
- TF-IDF treats words as independent; context and negation are not captured
- Class imbalance was partially addressed with `class_weight='balanced'` but SMOTE could further improve minority class recall

### Future Work

- Fine-tune **BERT** or **MentalBERT** for contextual understanding
- Apply **SMOTE** to oversample minority classes during training
- Expand to multilingual datasets
- Build a retraining pipeline to handle concept drift as language evolves

---

### References

- Inamdar et al. (2023). Machine learning driven mental stress detection on Reddit posts using NLP. *HumanCentric Intelligent Systems*, 3(2), 80–91.
- Muñoz et al. (2022). Comparative analysis of embedding techniques for mental health classification. *Healthcare Technology Letters*, 9(6), 152–161.
- Ji et al. (2021). MentalBERT: Publicly available pretrained language models for mental healthcare. *arXiv:2110.15621*.
- Zhang et al. (2022). NLP applied to mental illness detection: A narrative review. *NPJ Digital Medicine*, 5(1), 1–13.
- Sarkar, S. (2023). Sentiment Analysis for Mental Health [Dataset]. Kaggle.